# Cohort Preparation and Prediction Target

This notebook constructs the base cohort for the ICU mortality prediction project.

## Clinical prediction task

Use information available during the first 24 hours after ICU admission to predict:

```text
hospital_expire_flag
```

- `0`: survived the hospital admission
- `1`: died during the hospital admission

## Unit of analysis

One row represents:

```text
one hospital admission + its first ICU stay
```

A patient may contribute more than one hospital admission. Patient-level separation into training, validation, and test sets is performed later in Notebook 3.

## Why timestamps are retained

- `intime` defines the start of the feature window.
- `prediction_time` equals 24 hours after ICU admission.
- `outtime` is retained for cohort validation only.

`outtime`, hospital discharge information, length of stay, and death timestamps must not be used as model predictors.

## 1. Imports and paths

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_DIR = Path("../data/raw/mimic-iv-clinical-database-demo-2.2")
HOSP_DIR = DATA_DIR / "hosp"
ICU_DIR = DATA_DIR / "icu"
PROCESSED_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results/tables")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PATIENTS_PATH = HOSP_DIR / "patients.csv.gz"
ADMISSIONS_PATH = HOSP_DIR / "admissions.csv.gz"
ICUSTAYS_PATH = ICU_DIR / "icustays.csv.gz"

for path in [PATIENTS_PATH, ADMISSIONS_PATH, ICUSTAYS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

print("Data directory:", DATA_DIR)
print("All required files were found.")

Data directory: ..\data\raw\mimic-iv-clinical-database-demo-2.2
All required files were found.


## 2. Load the source tables

In [2]:
patients = pd.read_csv(PATIENTS_PATH)
admissions = pd.read_csv(ADMISSIONS_PATH)
icustays = pd.read_csv(ICUSTAYS_PATH)

print("Patients shape:", patients.shape)
print("Admissions shape:", admissions.shape)
print("ICU stays shape:", icustays.shape)

Patients shape: (100, 6)
Admissions shape: (275, 16)
ICU stays shape: (140, 8)


In [3]:
print("Patients columns:")
print(patients.columns.tolist())

print("\nAdmissions columns:")
print(admissions.columns.tolist())

print("\nICU stays columns:")
print(icustays.columns.tolist())

Patients columns:
['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']

Admissions columns:
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']

ICU stays columns:
['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los']


## 3. Validate source-table keys

- `subject_id`: patient identifier
- `hadm_id`: hospital admission identifier
- `stay_id`: ICU stay identifier

In [4]:
required_patient_columns = [
    "subject_id",
    "gender",
    "anchor_age",
]

required_admission_columns = [
    "subject_id",
    "hadm_id",
    "admittime",
    "dischtime",
    "deathtime",
    "admission_type",
    "admission_location",
    "insurance",
    "marital_status",
    "race",
    "hospital_expire_flag",
]

required_icu_columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "first_careunit",
    "intime",
    "outtime",
]

def check_required_columns(
    dataframe: pd.DataFrame,
    required_columns: list[str],
    table_name: str,
) -> None:
    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{table_name} is missing required columns: {missing_columns}"
        )

    print(f"{table_name}: all required columns are present.")


check_required_columns(
    patients,
    required_patient_columns,
    "patients",
)

check_required_columns(
    admissions,
    required_admission_columns,
    "admissions",
)

check_required_columns(
    icustays,
    required_icu_columns,
    "icustays",
)

patients: all required columns are present.
admissions: all required columns are present.
icustays: all required columns are present.


In [5]:
assert patients["subject_id"].notna().all()
assert patients["subject_id"].is_unique

assert admissions["subject_id"].notna().all()
assert admissions["hadm_id"].notna().all()
assert admissions["hadm_id"].is_unique

assert icustays["subject_id"].notna().all()
assert icustays["hadm_id"].notna().all()
assert icustays["stay_id"].notna().all()
assert icustays["stay_id"].is_unique

assert admissions["hospital_expire_flag"].notna().all()
assert admissions["hospital_expire_flag"].isin([0, 1]).all()

print("Source-table key validation passed.")

Source-table key validation passed.


In [6]:
source_summary = pd.DataFrame(
    [
        {
            "table": "patients",
            "rows": len(patients),
            "unique_patients": patients["subject_id"].nunique(),
            "unique_admissions": pd.NA,
            "unique_icu_stays": pd.NA,
        },
        {
            "table": "admissions",
            "rows": len(admissions),
            "unique_patients": admissions["subject_id"].nunique(),
            "unique_admissions": admissions["hadm_id"].nunique(),
            "unique_icu_stays": pd.NA,
        },
        {
            "table": "icustays",
            "rows": len(icustays),
            "unique_patients": icustays["subject_id"].nunique(),
            "unique_admissions": icustays["hadm_id"].nunique(),
            "unique_icu_stays": icustays["stay_id"].nunique(),
        },
    ]
)

source_summary

,table,rows,unique_patients,unique_admissions,unique_icu_stays
0,patients,100,100,<NA>,<NA>
1,admissions,275,100,275,<NA>
2,icustays,140,100,128,140


## 4. Convert timestamp columns

In [7]:
admission_time_columns = [
    "admittime",
    "dischtime",
    "deathtime",
]

icu_time_columns = [
    "intime",
    "outtime",
]

for column in admission_time_columns:
    admissions[column] = pd.to_datetime(
        admissions[column],
        errors="coerce",
    )

for column in icu_time_columns:
    icustays[column] = pd.to_datetime(
        icustays[column],
        errors="coerce",
    )

print(admissions[admission_time_columns].dtypes)
print(icustays[icu_time_columns].dtypes)

admittime    datetime64[us]
dischtime    datetime64[us]
deathtime    datetime64[us]
dtype: object
intime     datetime64[us]
outtime    datetime64[us]
dtype: object


In [8]:
timestamp_missingness = pd.Series(
    {
        "admittime_missing": admissions["admittime"].isna().sum(),
        "dischtime_missing": admissions["dischtime"].isna().sum(),
        "deathtime_missing": admissions["deathtime"].isna().sum(),
        "intime_missing": icustays["intime"].isna().sum(),
        "outtime_missing": icustays["outtime"].isna().sum(),
    },
    name="missing_count",
)

timestamp_missingness

admittime_missing      0
dischtime_missing      0
deathtime_missing    260
intime_missing         0
outtime_missing        0
Name: missing_count, dtype: int64

## 5. Examine the prediction target

In [9]:
target_summary_all_admissions = (
    admissions["hospital_expire_flag"]
    .value_counts(dropna=False)
    .rename_axis("hospital_expire_flag")
    .reset_index(name="count")
)

target_summary_all_admissions["label"] = (
    target_summary_all_admissions["hospital_expire_flag"]
    .map({0: "Survived", 1: "Died"})
)

target_summary_all_admissions["percentage"] = (
    target_summary_all_admissions["count"]
    / len(admissions)
    * 100
).round(2)

target_summary_all_admissions[
    ["hospital_expire_flag", "label", "count", "percentage"]
]

,hospital_expire_flag,label,count,percentage
0,0,Survived,260,94.55
1,1,Died,15,5.45


## 6. Select the first ICU stay within each hospital admission

Some hospital admissions contain multiple ICU stays. This first project version retains only the earliest ICU stay in each hospital admission.

Sorting uses:

1. `hadm_id`
2. ICU `intime`
3. `stay_id` as a deterministic tie-breaker

In [10]:
icu_stays_per_admission = (
    icustays.groupby("hadm_id")["stay_id"]
    .nunique()
    .sort_values(ascending=False)
)

print(
    "Maximum number of ICU stays in one admission:",
    int(icu_stays_per_admission.max()),
)

print(
    "Admissions with more than one ICU stay:",
    int((icu_stays_per_admission > 1).sum()),
)

icu_stays_per_admission.head(10)

Maximum number of ICU stays in one admission: 4
Admissions with more than one ICU stay: 9


hadm_id
23831430    4
27417763    3
22942076    2
22205327    2
23559586    2
28662225    2
27487226    2
22059910    2
28477280    2
20626031    1
Name: stay_id, dtype: int64

In [11]:
first_icu_stay = (
    icustays
    .sort_values(
        ["hadm_id", "intime", "stay_id"],
        na_position="last",
    )
    .drop_duplicates(
        subset="hadm_id",
        keep="first",
    )
    .copy()
)

print("All ICU stays:", len(icustays))
print("First ICU stays retained:", len(first_icu_stay))
print(
    "Unique admissions retained:",
    first_icu_stay["hadm_id"].nunique(),
)

All ICU stays: 140
First ICU stays retained: 128
Unique admissions retained: 128


In [12]:
assert first_icu_stay["hadm_id"].is_unique
assert first_icu_stay["stay_id"].is_unique

missing_first_icu_intime = first_icu_stay["intime"].isna().sum()

print("Missing intime values in selected ICU stays:", missing_first_icu_intime)

if missing_first_icu_intime > 0:
    raise ValueError(
        "Selected first ICU stays contain missing intime values. "
        "The 24-hour feature window cannot be defined."
    )

print("First-ICU-stay validation passed.")

Missing intime values in selected ICU stays: 0
First-ICU-stay validation passed.


## 7. Merge first ICU stays with hospital admissions

In [13]:
cohort = first_icu_stay.merge(
    admissions,
    on=["subject_id", "hadm_id"],
    how="inner",
    validate="one_to_one",
    suffixes=("_icu", "_admission"),
)

print("Cohort shape after ICU-admission merge:", cohort.shape)
cohort.head()

Cohort shape after ICU-admission merge: (128, 22)


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10023771,20044587,33177122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2113-08-25 09:32:41,2113-08-27 16:27:53,2.288333,2113-08-25 07:15:00,2113-08-30 14:15:00,NaT,ELECTIVE,P47E1G,PHYSICIAN REFERRAL,SKILLED NURSING FACILITY,Medicare,ENGLISH,MARRIED,WHITE,NaN,NaN,0
1,10005909,20199380,36496303,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2144-10-29 23:09:03,2144-11-02 15:24:29,3.677384,2144-10-28 23:20:00,2144-11-02 15:23:00,NaT,OBSERVATION ADMIT,P43BTJ,EMERGENCY ROOM,HOME,Other,ENGLISH,MARRIED,WHITE,2144-10-28 18:29:00,2144-10-29 00:10:00,0
2,10003400,20214994,32128372,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2137-02-25 23:37:19,2137-03-10 21:29:36,12.911308,2137-02-24 10:00:00,2137-03-19 15:45:00,NaT,URGENT,P60ZCO,TRANSFER FROM SKILLED NURSING FACILITY,CHRONIC/LONG TERM ACUTE CARE,Medicare,ENGLISH,MARRIED,BLACK/AFRICAN AMERICAN,NaN,NaN,0
3,10008454,20291550,31959184,Trauma SICU (TSICU),Trauma SICU (TSICU),2110-11-30 17:11:36,2110-12-05 16:48:24,4.983889,2110-11-30 06:31:00,2110-12-10 15:53:00,NaT,EW EMER.,P77BSD,EMERGENCY ROOM,HOME HEALTH CARE,Other,ENGLISH,SINGLE,WHITE,2110-11-30 04:45:00,2110-11-30 08:03:00,0
4,10019385,20297618,39268883,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2180-02-21 08:34:06,2180-02-22 16:05:14,1.313287,2180-02-15 20:28:00,2180-02-25 13:45:00,NaT,URGENT,P536JC,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Other,ENGLISH,MARRIED,WHITE,NaN,NaN,0


## 8. Add patient demographics

In [14]:
cohort = cohort.merge(
    patients,
    on="subject_id",
    how="left",
    validate="many_to_one",
)

print("Cohort shape after patient merge:", cohort.shape)
cohort.head()

Cohort shape after patient merge: (128, 27)


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10023771,20044587,33177122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2113-08-25 09:32:41,2113-08-27 16:27:53,2.288333,2113-08-25 07:15:00,2113-08-30 14:15:00,NaT,ELECTIVE,P47E1G,PHYSICIAN REFERRAL,SKILLED NURSING FACILITY,Medicare,ENGLISH,MARRIED,WHITE,NaN,NaN,0,M,70,2113,2011 - 2013,NaN
1,10005909,20199380,36496303,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2144-10-29 23:09:03,2144-11-02 15:24:29,3.677384,2144-10-28 23:20:00,2144-11-02 15:23:00,NaT,OBSERVATION ADMIT,P43BTJ,EMERGENCY ROOM,HOME,Other,ENGLISH,MARRIED,WHITE,2144-10-28 18:29:00,2144-10-29 00:10:00,0,F,40,2144,2014 - 2016,NaN
2,10003400,20214994,32128372,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2137-02-25 23:37:19,2137-03-10 21:29:36,12.911308,2137-02-24 10:00:00,2137-03-19 15:45:00,NaT,URGENT,P60ZCO,TRANSFER FROM SKILLED NURSING FACILITY,CHRONIC/LONG TERM ACUTE CARE,Medicare,ENGLISH,MARRIED,BLACK/AFRICAN AMERICAN,NaN,NaN,0,F,72,2134,2011 - 2013,2137-09-02
3,10008454,20291550,31959184,Trauma SICU (TSICU),Trauma SICU (TSICU),2110-11-30 17:11:36,2110-12-05 16:48:24,4.983889,2110-11-30 06:31:00,2110-12-10 15:53:00,NaT,EW EMER.,P77BSD,EMERGENCY ROOM,HOME HEALTH CARE,Other,ENGLISH,SINGLE,WHITE,2110-11-30 04:45:00,2110-11-30 08:03:00,0,F,26,2110,2011 - 2013,NaN
4,10019385,20297618,39268883,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2180-02-21 08:34:06,2180-02-22 16:05:14,1.313287,2180-02-15 20:28:00,2180-02-25 13:45:00,NaT,URGENT,P536JC,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Other,ENGLISH,MARRIED,WHITE,NaN,NaN,0,M,44,2180,2014 - 2016,NaN


## 9. Apply cohort inclusion criteria

In [15]:
cohort_before_filtering = cohort.copy()

inclusion_mask = (
    cohort["subject_id"].notna()
    & cohort["hadm_id"].notna()
    & cohort["stay_id"].notna()
    & cohort["intime"].notna()
    & cohort["hospital_expire_flag"].notna()
    & cohort["anchor_age"].ge(18)
)

cohort = cohort.loc[inclusion_mask].copy()

filter_summary = pd.Series(
    {
        "rows_before_filtering": len(cohort_before_filtering),
        "rows_after_filtering": len(cohort),
        "rows_removed": (
            len(cohort_before_filtering) - len(cohort)
        ),
        "patients_after_filtering": cohort["subject_id"].nunique(),
        "admissions_after_filtering": cohort["hadm_id"].nunique(),
        "icu_stays_after_filtering": cohort["stay_id"].nunique(),
    },
    name="value",
)

filter_summary

rows_before_filtering         128
rows_after_filtering          128
rows_removed                    0
patients_after_filtering      100
admissions_after_filtering    128
icu_stays_after_filtering     128
Name: value, dtype: int64

## 10. Define the prediction time

In [16]:
cohort["prediction_time"] = (
    cohort["intime"] + pd.Timedelta(hours=24)
)

cohort[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "intime",
        "prediction_time",
        "outtime",
    ]
].head()

,subject_id,hadm_id,stay_id,intime,prediction_time,outtime
0,10023771,20044587,33177122,2113-08-25 09:32:41,2113-08-26 09:32:41,2113-08-27 16:27:53
1,10005909,20199380,36496303,2144-10-29 23:09:03,2144-10-30 23:09:03,2144-11-02 15:24:29
2,10003400,20214994,32128372,2137-02-25 23:37:19,2137-02-26 23:37:19,2137-03-10 21:29:36
3,10008454,20291550,31959184,2110-11-30 17:11:36,2110-12-01 17:11:36,2110-12-05 16:48:24
4,10019385,20297618,39268883,2180-02-21 08:34:06,2180-02-22 08:34:06,2180-02-22 16:05:14


The intended predictor window is:

```text
intime <= feature timestamp <= prediction_time
```

The outcome may occur later during the same hospital admission.

## 11. Validate temporal relationships

In [17]:
temporal_validation = pd.Series(
    {
        "icu_out_before_icu_in": int(
            (
                cohort["outtime"].notna()
                & (cohort["outtime"] < cohort["intime"])
            ).sum()
        ),
        "hospital_discharge_before_admission": int(
            (
                cohort["dischtime"].notna()
                & cohort["admittime"].notna()
                & (cohort["dischtime"] < cohort["admittime"])
            ).sum()
        ),
        "icu_in_before_hospital_admission": int(
            (
                cohort["admittime"].notna()
                & (cohort["intime"] < cohort["admittime"])
            ).sum()
        ),
        "icu_in_after_hospital_discharge": int(
            (
                cohort["dischtime"].notna()
                & (cohort["intime"] > cohort["dischtime"])
            ).sum()
        ),
    },
    name="invalid_row_count",
)

temporal_validation

icu_out_before_icu_in                  0
hospital_discharge_before_admission    0
icu_in_before_hospital_admission       2
icu_in_after_hospital_discharge        0
Name: invalid_row_count, dtype: int64

In [18]:
assert temporal_validation["icu_out_before_icu_in"] == 0
assert temporal_validation["hospital_discharge_before_admission"] == 0
assert temporal_validation["icu_in_before_hospital_admission"] == 0
assert temporal_validation["icu_in_after_hospital_discharge"] == 0

print("Temporal validation passed.")

AssertionError: 

## 12. Validate the final cohort

In [ ]:
assert cohort["subject_id"].notna().all()
assert cohort["hadm_id"].notna().all()
assert cohort["stay_id"].notna().all()
assert cohort["intime"].notna().all()
assert cohort["prediction_time"].notna().all()

assert cohort["hadm_id"].is_unique
assert cohort["stay_id"].is_unique

assert cohort["hospital_expire_flag"].notna().all()
assert cohort["hospital_expire_flag"].isin([0, 1]).all()

assert cohort["anchor_age"].ge(18).all()
assert (cohort["prediction_time"] > cohort["intime"]).all()

print("Final cohort validation passed.")

In [ ]:
cohort_summary = pd.Series(
    {
        "rows": len(cohort),
        "unique_patients": cohort["subject_id"].nunique(),
        "unique_admissions": cohort["hadm_id"].nunique(),
        "unique_icu_stays": cohort["stay_id"].nunique(),
        "deaths": int(cohort["hospital_expire_flag"].sum()),
        "mortality_rate": cohort["hospital_expire_flag"].mean(),
        "minimum_age": cohort["anchor_age"].min(),
        "maximum_age": cohort["anchor_age"].max(),
    },
    name="value",
)

cohort_summary

## 13. Examine mortality in the ICU cohort

In [ ]:
mortality_summary = (
    cohort["hospital_expire_flag"]
    .value_counts(dropna=False)
    .rename_axis("hospital_expire_flag")
    .reset_index(name="count")
)

mortality_summary["label"] = (
    mortality_summary["hospital_expire_flag"]
    .map({0: "Survived", 1: "Died"})
)

mortality_summary["percentage"] = (
    mortality_summary["count"] / len(cohort) * 100
).round(2)

mortality_summary[
    ["hospital_expire_flag", "label", "count", "percentage"]
]

## 14. Identify leakage variables

The following variables may be retained for cohort construction or validation, but they must not be used as predictors:

- `hospital_expire_flag`: target
- `deathtime`: directly reveals death
- `dod`: directly reveals death
- `dischtime`: only known at the end of hospitalization
- `discharge_location`: only known at discharge
- `outtime`: only known after the ICU stay
- `los`: completed ICU length of stay
- `prediction_time`: feature-window boundary, not a clinical predictor

`intime` is retained to construct time-based features and join clinical events. The raw timestamp itself will not be used directly as a predictor.

In [ ]:
leakage_or_nonpredictor_columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "hospital_expire_flag",
    "deathtime",
    "dod",
    "dischtime",
    "discharge_location",
    "outtime",
    "los",
    "prediction_time",
]

leakage_or_nonpredictor_columns

## 15. Create the base modeling cohort

The saved file preserves identifiers and timestamps required for:

- patient-level splitting
- event-table joins
- first-24-hour feature extraction
- data validation

The later model predictor matrix will exclude identifiers and leakage variables.

In [ ]:
selected_columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "intime",
    "prediction_time",
    "outtime",
    "gender",
    "anchor_age",
    "admission_type",
    "admission_location",
    "insurance",
    "marital_status",
    "race",
    "first_careunit",
    "hospital_expire_flag",
]

missing_selected_columns = [
    column
    for column in selected_columns
    if column not in cohort.columns
]

if missing_selected_columns:
    raise ValueError(
        f"Missing selected columns: {missing_selected_columns}"
    )

modeling_df = cohort[selected_columns].copy()

print("Base modeling cohort shape:", modeling_df.shape)
modeling_df.head()

In [ ]:
assert modeling_df["hadm_id"].is_unique
assert modeling_df["stay_id"].is_unique
assert modeling_df["intime"].notna().all()
assert modeling_df["prediction_time"].notna().all()
assert modeling_df["hospital_expire_flag"].isin([0, 1]).all()

print("Base modeling cohort validation passed.")

## 16. Review missingness and categorical values

In [ ]:
missingness_summary = pd.DataFrame(
    {
        "missing_count": modeling_df.isna().sum(),
        "missing_percentage": (
            modeling_df.isna().mean() * 100
        ).round(2),
        "data_type": modeling_df.dtypes.astype(str),
    }
).sort_values(
    "missing_percentage",
    ascending=False,
)

missingness_summary

In [ ]:
categorical_columns = [
    "gender",
    "admission_type",
    "admission_location",
    "insurance",
    "marital_status",
    "race",
    "first_careunit",
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(
        modeling_df[column]
        .value_counts(dropna=False)
    )

## 17. Save the cohort and summary tables

In [ ]:
cohort_output_path = (
    PROCESSED_DIR / "icu_mortality_cohort_demo.csv"
)

cohort_summary_path = (
    RESULTS_DIR / "cohort_summary.csv"
)

mortality_summary_path = (
    RESULTS_DIR / "cohort_mortality_summary.csv"
)

missingness_summary_path = (
    RESULTS_DIR / "cohort_missingness_summary.csv"
)

temporal_validation_path = (
    RESULTS_DIR / "cohort_temporal_validation.csv"
)

modeling_df.to_csv(
    cohort_output_path,
    index=False,
)

cohort_summary.to_csv(
    cohort_summary_path,
    header=True,
)

mortality_summary.to_csv(
    mortality_summary_path,
    index=False,
)

missingness_summary.to_csv(
    missingness_summary_path,
)

temporal_validation.to_csv(
    temporal_validation_path,
    header=True,
)

print("Saved:")
print("-", cohort_output_path)
print("-", cohort_summary_path)
print("-", mortality_summary_path)
print("-", missingness_summary_path)
print("-", temporal_validation_path)

## 18. Reload and validate the saved file

In [ ]:
saved_cohort = pd.read_csv(
    cohort_output_path,
    parse_dates=[
        "intime",
        "prediction_time",
        "outtime",
    ],
)

assert len(saved_cohort) == len(modeling_df)
assert saved_cohort["hadm_id"].is_unique
assert saved_cohort["stay_id"].is_unique
assert saved_cohort["intime"].notna().all()
assert saved_cohort["prediction_time"].notna().all()
assert saved_cohort["hospital_expire_flag"].isin([0, 1]).all()

expected_time_difference = pd.Timedelta(hours=24)

assert (
    saved_cohort["prediction_time"]
    - saved_cohort["intime"]
).eq(expected_time_difference).all()

print("Saved file validation passed.")
print("Output path:", cohort_output_path)
print("Saved shape:", saved_cohort.shape)
print("Saved columns:")
print(saved_cohort.columns.tolist())

## Notebook 2 Summary

- Loaded `patients`, `admissions`, and `icustays`.
- Validated source-table keys and the mortality target.
- Selected the first ICU stay within each hospital admission.
- Added patient demographic information.
- Restricted the cohort to adults with valid identifiers and ICU admission times.
- Defined `prediction_time` as 24 hours after ICU admission.
- Checked temporal relationships.
- Retained `intime`, `prediction_time`, and `outtime` for later feature extraction and validation.
- Saved one row per hospital admission and first ICU stay.

## Next step

Run Notebook 3 again so the patient-level train, validation, and test files inherit the timestamp columns.

Then create Notebook 4 to:

1. engineer admission-time features;
2. join ICU event data;
3. extract first-24-hour vital signs.